In [0]:
%sql

use catalog trueanalytics_data;

In [0]:
import pyspark
import pyspark.sql.functions as F
import pyspark.sql.types as T 
from functools import partial
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import lit

In [0]:
def save_to_parquet(df, save_path):
    (df.write.format('parquet')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save parquet to ", save_path)


In [0]:
# parameter: par_month
dbutils.widgets.text("par_month", "202510")
par_month = dbutils.widgets.get("par_month")

try:
  par_month = int(par_month)
except ValueError:
  par_month = 0
  raise ValueError("par_month value must be numeric")

if par_month!=0:
  pass
else:
  dbutils.notebook.exit("Aborting as ondition not met. Further tasks will be skipped")

# customer360 date
par_month_obj = datetime.strptime(str(par_month), '%Y%m')
next_par_month_obj = par_month_obj + relativedelta(months=1)
cust360_date = int(next_par_month_obj.strftime('%Y%m') + '01')

# debug
display('par_month = ',par_month)
display(' customer360_date = ',cust360_date)


In [0]:
save_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/{par_month}_360_feature.parquet'
nantional_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/country_group_chula.csv'
home_region_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/home_region_chula.csv'

In [0]:
prep_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/{par_month}_footfall.parquet'
df = spark.read.parquet(prep_path)\
    .withColumn('par_month', lit(par_month))\
        .select('msisdn'
                ,'a_country_name'
                ,'geog_resident_location_v1_province_en_cat'
                ,'geog_resident_location_v1_sub_district_en_cat'
                ,'geog_resident_location_v1_district_en_cat'
                ,'geog_work_location_v1_province_en_cat'
                ,'geog_work_location_v1_sub_district_en_cat'
                ,'geog_work_location_v1_district_en_cat'
                ,'demo_tourist_sim_v1_tourist_bin'
                ,'roaming_flag'
                ,'is_bmr'
                ,'is_non_bmr'
                ,'is_foriegner'
                ).distinct()
df.count()
# prep_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/{par_month}_footfall.parquet'
# df = spark.read.parquet(prep_path)\
#     .withColumn('par_month', lit(par_month))\
#         .select('msisdn'
#                 ,'a_country_name'
#                 ,'geog_resident_location_v1_province_en_cat'
#                 ,'geog_resident_location_sub_district_en_cat'
#                 ,'geog_resident_location_district_en_cat'
#                 ,'geog_work_location_province_en_cat'
#                 ,'geog_work_location_sub_district_en_cat'
#                 ,'geog_work_location_district_en_cat'
#                 ,'demo_tourist_sim_v1_tourist_bin'
#                 ,'roaming_flag'
#                 ,'is_bmr'
#                 ,'is_non_bmr'
#                 ,'is_foriegner'
#                 ).distinct()
# df.count()

# Customer360

In [0]:
columns = [
'msisdn',
'demo_gender_fog_v1_final_gender_bin', # gender
'demo_affluence_grid_v1_affluence_group_bin', # affluence
'demo_age_fact_v2_final_age_num', # age_number
'inte_fashion_apparel_v1_fashion_apparel_v1_cat', # Fashion
'inte_cosmetic_surgery_v1_cosmetic_surgery_v1_cat', # Cosmetic
'inte_fashion_cosmetic_store_v2_fashion_cosmetic_store_v2_cat', # Cosmetic
'inte_cosmetic_online_store_v1_cosmetic_online_store_v1_cat', # Cosmetic
'inte_cosmetic_store_v1_cosmetic_store_v1_cat', # Cosmetic
'inte_ecommerce_consumer_electronics_brand_v1_ecommerce_consumer_electronics_brand_v1_cat', # Ecommerce
'inte_ecommerce_consumer_electronics_retail_v1_ecommerce_consumer_electronics_retail_v1_cat', # Ecommerce
'inte_food_chain_restuarant_v1_food_chain_restuarant_v1_cat', # Food and Beverage
'inte_food_junkfood_lover_v3_food_junkfood_lover_v3_cat', # Food and Beverage
'inte_chain_restuarant_chain_restuarant_cat', # Food and Beverage
'inte_highend_japanese_food_v1_highend_japanese_food_v1_cat', # Food and Beverage
'inte_luxury_hotel_thailand_v1_luxury_hotel_thailand_v1_cat', # Hotel
'inte_lifestage_parents_v2_lifestage_parents_v2_cat', # Lifestage
'inte_lifestage_pregnancy_v1_lifestage_pregnancy_v1_cat', # Lifestage
'inte_lifestage_kids_5_8yo_v1_lifestage_kids_5_8yo_v1_cat', # Lifestage
'inte_lifestage_baby_v1_lifestage_baby_v1_cat', # Lifestage
'inte_lifestage_toddler_v1_lifestage_toddler_v1_cat', # Lifestage
'inte_healthcare_gym_v1_healthcare_gym_v1_cat', # Health and Fitness
'inte_sport_bicycle_v1_sport_bicycle_v1_cat', # Sports
'inte_sport_running_v1_sport_running_v1_cat', # Sports
'inte_sport_badminton_v1_sport_badminton_v1_cat', # Sports
'inte_sport_basketball_v1_sport_basketball_v1_cat', # Sports
'inte_basketball_content_basketball_content_cat', # Sports
'inte_sport_boxing_v1_sport_boxing_v1_cat', # Sports
'inte_sport_shooting_v1_sport_shooting_v1_cat', # Sports
'inte_sport_tennis_v1_sport_tennis_v1_cat', # Sports
'inte_sport_volleyball_v1_sport_volleyball_v1_cat', # Sports
'inte_sport_fishing_v1_sport_fishing_v1_cat', # Sports
'inte_sport_football_lover_v2_sport_football_lover_v2_cat', # Sports
'inte_sport_golf_v1_sport_golf_v1_cat', # Sports
'inte_sport_mixsport_v1_sport_mixsport_v1_cat', # Sports
'inte_movie_ticket_v1_movie_ticket_v1_cat', # Movies and Entertainment
'inte_entertainment_cinema_v1_entertainment_cinema_v1_cat', # Movies and Entertainment
'inte_pet_lover_v1_pet_lover_v1_cat', # Pets
'inte_healthcare_health_conditions_and_concerns_v1_healthcare_health_conditions_and_concerns_v1_cat', # Health and Fitness
'inte_healthcare_content_healthcare_content_cat', # Health and Fitness
'inte_healthcare_hospital_v1_healthcare_hospital_v1_cat', # Health and Fitness
'inte_healthcare_nutrition_and_fittness_v1_healthcare_nutrition_and_fittness_v1_cat', # Health and Fitness
'inte_concert_ticket_v1_concert_ticket_v1_cat', # Movies and Entertainment
'inte_education_university_student_v1_education_university_student_v1_cat', # Education
'inte_education_international_school_v1_education_international_school_v1_cat', # Education
'inte_education_university_website_v1_education_university_website_v1_cat', # Education
'beha_credit_card_app_usage_v1_credit_card_app_usage_v1_cat', # Finance and Banking
'inte_finance_credit_v1_finance_credit_v1_cat', # Finance and Banking
'inte_insurance_insurance_nonlife_v2_insurance_insurance_nonlife_v2_cat', # Insurance
'inte_securities_broker_site_v1_securities_broker_site_v1_cat', # Insurance
'inte_securities_broker_site_v2_securities_broker_site_v2_cat', # Insurance
'inte_realestate_real_estate_developer_v3_realestate_real_estate_developer_v3_cat', # Real estate
'inte_chinese_web_v1_chinese_web_v1_cat', # Tourist
]

In [0]:
df_cust360 = spark.read.table('trueanalytics_data.customer360.customer360_snapshot')\
    .filter((F.col('par_day')==cust360_date) & (F.col('activated_flag')=='1'))\
    .select(columns)

In [0]:
df_mall_cust360 = df.join(df_cust360, on='msisdn', how='left')

In [0]:
df_intermediate_gender = df_mall_cust360.withColumnRenamed('demo_gender_fog_v1_final_gender_bin','gender')
df_intermediate_gender = df_intermediate_gender.withColumn('gender', 
    F.when(F.col('gender').isNull(), F.lit('U'))
    .when(F.col('gender') == 'female', F.lit('F'))
    .when(F.col('gender') == 'male', F.lit('M'))
    .otherwise(F.col('gender'))
)

In [0]:
df_intermediate_age = df_intermediate_gender.withColumn("demo_age_fact_v2_final_age_num", F.col("demo_age_fact_v2_final_age_num").cast(T.DoubleType()))

df_intermediate_age = df_intermediate_age.withColumn("age_range", 
  F.when(F.col("demo_age_fact_v2_final_age_num") <= 12, '1_12')
  .when(F.col("demo_age_fact_v2_final_age_num").between(13, 17), '13_17')
  .when(F.col("demo_age_fact_v2_final_age_num").between(18, 24), '18_24')
  .when(F.col("demo_age_fact_v2_final_age_num").between(25, 34), '25_34')
  .when(F.col("demo_age_fact_v2_final_age_num").between(35, 44), '35_44')
  .when(F.col("demo_age_fact_v2_final_age_num").between(45, 54), '45_54')
  .when(F.col("demo_age_fact_v2_final_age_num").between(55, 59), '55_59')
  .when(F.col("demo_age_fact_v2_final_age_num").between(60, 64), '60_64')
  .when(F.col("demo_age_fact_v2_final_age_num")>=65, '>=65')
  .when(F.col('demo_age_fact_v2_final_age_num').isNull(), 'unidentified')
)

# debug
# display(df_intermediate_age.select('age_range').distinct())

In [0]:
# rename
df_intermediate_ = df_intermediate_age.withColumnRenamed("geog_resident_location_v1_province_en_cat", "home_province")
df_intermediate_ = df_intermediate_.withColumnRenamed("geog_resident_location_v1_district_en_cat", "home_district")
df_intermediate_ = df_intermediate_.withColumnRenamed("geog_resident_location_v1_sub_district_en_cat", "home_subdistrict")
df_intermediate_ = df_intermediate_.withColumnRenamed("geog_work_location_v1_province_en_cat", "work_province")
df_intermediate_ = df_intermediate_.withColumnRenamed("geog_work_location_v1_district_en_cat", "work_district")
df_intermediate_ = df_intermediate_.withColumnRenamed("geog_work_location_v1_sub_district_en_cat", "work_subdistrict")

# lowercase
df_intermediate_ = df_intermediate_.withColumn("home_province", F.lower(F.col("home_province")))
df_intermediate_ = df_intermediate_.withColumn("home_district", F.lower(F.col("home_district")))
df_intermediate_ = df_intermediate_.withColumn("home_subdistrict", F.lower(F.col("home_subdistrict")))
df_intermediate_ = df_intermediate_.withColumn("work_province", F.lower(F.col("work_province")))
df_intermediate_ = df_intermediate_.withColumn("work_district", F.lower(F.col("work_district")))
df_intermediate_ = df_intermediate_.withColumn("work_subdistrict", F.lower(F.col("work_subdistrict")))


df_intermediate_homework = df_intermediate_.withColumn("home_subdistrict", 
    F.when((F.col("home_province")=='bangkok') & (F.col("home_subdistrict")=='chantharakasem'), F.lit('chan kasem'))
    .otherwise(F.col('home_subdistrict'))
)
df_intermediate_homework = df_intermediate_.withColumn("work_subdistrict", 
    F.when((F.col("work_province")=='bangkok') & (F.col("work_subdistrict")=='chantharakasem'), F.lit('chan kasem'))
    .otherwise(F.col('work_subdistrict'))
)
df_home_region = (spark.read
      .option("header", "true").option("inferSchema", "true")
      .csv(home_region_path))
df_home_region = df_home_region.withColumn("home_province", F.regexp_replace(F.col("home_province"), " ", ""))

df_intermediate_region = df_intermediate_homework.join(df_home_region, on='home_province', how='left')

# Array of Interest
- interest	The category or type of data being collected or classified, often related to user interests or actions. and partiotion in index is [art_and_culture , education, investment, ecommerce, games, sport, social_media, food_and_drink]

In [0]:
# Fashion / 
# Cosmetic Surgery / 
# Cosmetic / 
# F&B / 
# Highend F&B / 
# Ecommerce
df_intermediate_interest = df_intermediate_region.withColumn("interest_IS", 
    F.array(
        # flag[0]: Fashion
        F.when(F.col("inte_fashion_apparel_v1_fashion_apparel_v1_cat").isNotNull(), 1).otherwise(0),  
        # flag[1]: Cosmetic Surgery 
        F.when(F.col('inte_cosmetic_surgery_v1_cosmetic_surgery_v1_cat').isNotNull(), 1).otherwise(0),     
        # flag[2]: Cosmetic
        F.when((F.col('inte_cosmetic_surgery_v1_cosmetic_surgery_v1_cat').isNotNull())|
               (F.col('inte_fashion_cosmetic_store_v2_fashion_cosmetic_store_v2_cat').isNotNull())|
               (F.col('inte_cosmetic_online_store_v1_cosmetic_online_store_v1_cat').isNotNull())|
               (F.col('inte_cosmetic_store_v1_cosmetic_store_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[4]: F&B
        F.when((F.col('inte_food_chain_restuarant_v1_food_chain_restuarant_v1_cat').isNotNull())|
               (F.col('inte_food_junkfood_lover_v3_food_junkfood_lover_v3_cat').isNotNull())|
               (F.col('inte_chain_restuarant_chain_restuarant_cat').isNotNull()), 1).otherwise(0), 
        # flag[5]: Highend F&B
        F.when((F.col('inte_highend_japanese_food_v1_highend_japanese_food_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[3]: Ecommerce
        F.when((F.col('inte_ecommerce_consumer_electronics_brand_v1_ecommerce_consumer_electronics_brand_v1_cat').isNotNull())|
               (F.col('inte_ecommerce_consumer_electronics_retail_v1_ecommerce_consumer_electronics_retail_v1_cat').isNotNull()), 1).otherwise(0)
    )
)                                             

In [0]:
# Gym users /
# Sports / 
# Health and Wellness / 
# Hotel / 
# Cinema / 
# Live Concerts
df_intermediate_interest = df_intermediate_interest.withColumn("interest_IL", 
    F.array(
        # flag[8]: Gym users  
        F.when((F.col('inte_healthcare_gym_v1_healthcare_gym_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[9]: Sports
        F.when((F.col('inte_sport_bicycle_v1_sport_bicycle_v1_cat').isNotNull())|
               (F.col('inte_sport_running_v1_sport_running_v1_cat').isNotNull())|
               (F.col('inte_sport_badminton_v1_sport_badminton_v1_cat').isNotNull())|
               (F.col('inte_sport_basketball_v1_sport_basketball_v1_cat').isNotNull())|
               (F.col('inte_basketball_content_basketball_content_cat').isNotNull())|
               (F.col('inte_sport_boxing_v1_sport_boxing_v1_cat').isNotNull())|
               (F.col('inte_sport_shooting_v1_sport_shooting_v1_cat').isNotNull())|
               (F.col('inte_sport_tennis_v1_sport_tennis_v1_cat').isNotNull())|
               (F.col('inte_sport_volleyball_v1_sport_volleyball_v1_cat').isNotNull())|
               (F.col('inte_sport_fishing_v1_sport_fishing_v1_cat').isNotNull())|
               (F.col('inte_sport_football_lover_v2_sport_football_lover_v2_cat').isNotNull())|
               (F.col('inte_sport_golf_v1_sport_golf_v1_cat').isNotNull())|
               (F.col('inte_sport_mixsport_v1_sport_mixsport_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[12]: Health and Wellness
        F.when((F.col('inte_healthcare_health_conditions_and_concerns_v1_healthcare_health_conditions_and_concerns_v1_cat').isNotNull())|
               (F.col('inte_healthcare_content_healthcare_content_cat').isNotNull())|
               (F.col('inte_healthcare_hospital_v1_healthcare_hospital_v1_cat').isNotNull())|
               (F.col('inte_healthcare_nutrition_and_fittness_v1_healthcare_nutrition_and_fittness_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[6]: Hotel
        F.when((F.col('inte_luxury_hotel_thailand_v1_luxury_hotel_thailand_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[10]: Cinema
        F.when((F.col('inte_movie_ticket_v1_movie_ticket_v1_cat').isNotNull())|
               (F.col('inte_entertainment_cinema_v1_entertainment_cinema_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[13]: Live Concerts
        F.when((F.col('inte_concert_ticket_v1_concert_ticket_v1_cat').isNotNull()), 1).otherwise(0)
    )
)

In [0]:
# "Family & Lifestage" Interest 4 หมวด > 
# Pet Lovers / 
# Parents / 
# University and International students /
# Financial and Investment
df_intermediate_interest = df_intermediate_interest.withColumn("interest_IF", 
    F.array(
        # flag[11]: Pet Lovers
        F.when((F.col('inte_pet_lover_v1_pet_lover_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[7]: Parents
        F.when((F.col('inte_lifestage_parents_v2_lifestage_parents_v2_cat').isNotNull())|
               (F.col('inte_lifestage_pregnancy_v1_lifestage_pregnancy_v1_cat').isNotNull())|
               (F.col('inte_lifestage_kids_5_8yo_v1_lifestage_kids_5_8yo_v1_cat').isNotNull())|
               (F.col('inte_lifestage_baby_v1_lifestage_baby_v1_cat').isNotNull())|
               (F.col('inte_lifestage_toddler_v1_lifestage_toddler_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[14]: University and International students
        F.when((F.col('inte_education_university_student_v1_education_university_student_v1_cat').isNotNull())|
               (F.col('inte_education_international_school_v1_education_international_school_v1_cat').isNotNull())|
               (F.col('inte_education_university_website_v1_education_university_website_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[15]: Financial and Investment
        F.when((F.col('beha_credit_card_app_usage_v1_credit_card_app_usage_v1_cat').isNotNull())|
               (F.col('inte_finance_credit_v1_finance_credit_v1_cat').isNotNull())|
               (F.col('inte_insurance_insurance_nonlife_v2_insurance_insurance_nonlife_v2_cat').isNotNull())|
               (F.col('inte_securities_broker_site_v1_securities_broker_site_v1_cat').isNotNull())|
               (F.col('inte_securities_broker_site_v2_securities_broker_site_v2_cat').isNotNull())|
               (F.col('inte_realestate_real_estate_developer_v3_realestate_real_estate_developer_v3_cat').isNotNull()), 1).otherwise(0) 
    )
)



In [0]:
# Report แสดง ภาพรวม Interest รวมกลุ่มเป็น 9 หมวด
# 1. "Appearance" ประกอบด้วย Fashion / Cosmetic Surgery / Cosmetic
# 2. "Food" ประกอบด้วย F&B / Highend F&B
# 3. "Hotel" ประกอบด้วย Hotel
# 4. "Sport and Wellness" ประกอบด้วย Gym users / Sports / Health and Wellness
# 5. "Parents"
# 6. "University and International students"
# 7. "Digital & Financial" ประกอบด้วย Ecommerce / Financial and Investment
# 8. "Entertainment" ประกอบด้วย Cinema / Live Concerts
# 9. "Pet Lovers"

df_intermediate_interest = df_intermediate_interest.withColumn("interest_I", 
    F.array(
        # flag[0]: Fashion + Cosmetic Surgery + Cosmetic
        F.when((F.col("inte_fashion_apparel_v1_fashion_apparel_v1_cat").isNotNull()) |
               (F.col('inte_cosmetic_surgery_v1_cosmetic_surgery_v1_cat').isNotNull())|
               (F.col('inte_fashion_cosmetic_store_v2_fashion_cosmetic_store_v2_cat').isNotNull())|
               (F.col('inte_cosmetic_online_store_v1_cosmetic_online_store_v1_cat').isNotNull())|
               (F.col('inte_cosmetic_store_v1_cosmetic_store_v1_cat').isNotNull()), 1).otherwise(0), 
        # flag[1]: F&B + Highend F&B
        F.when((F.col('inte_food_chain_restuarant_v1_food_chain_restuarant_v1_cat').isNotNull())|
               (F.col('inte_food_junkfood_lover_v3_food_junkfood_lover_v3_cat').isNotNull())|
               (F.col('inte_chain_restuarant_chain_restuarant_cat').isNotNull())|
               (F.col('inte_highend_japanese_food_v1_highend_japanese_food_v1_cat').isNotNull()), 1).otherwise(0), 
        # flag[2]: Hotel
        F.when((F.col('inte_luxury_hotel_thailand_v1_luxury_hotel_thailand_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[3]: Gym users  + Sports + Health and Wellness
        F.when((F.col('inte_healthcare_gym_v1_healthcare_gym_v1_cat').isNotNull())|
               (F.col('inte_sport_bicycle_v1_sport_bicycle_v1_cat').isNotNull())|
               (F.col('inte_sport_running_v1_sport_running_v1_cat').isNotNull())|
               (F.col('inte_sport_badminton_v1_sport_badminton_v1_cat').isNotNull())|
               (F.col('inte_sport_basketball_v1_sport_basketball_v1_cat').isNotNull())|
               (F.col('inte_basketball_content_basketball_content_cat').isNotNull())|
               (F.col('inte_sport_boxing_v1_sport_boxing_v1_cat').isNotNull())|
               (F.col('inte_sport_shooting_v1_sport_shooting_v1_cat').isNotNull())|
               (F.col('inte_sport_tennis_v1_sport_tennis_v1_cat').isNotNull())|
               (F.col('inte_sport_volleyball_v1_sport_volleyball_v1_cat').isNotNull())|
               (F.col('inte_sport_fishing_v1_sport_fishing_v1_cat').isNotNull())|
               (F.col('inte_sport_football_lover_v2_sport_football_lover_v2_cat').isNotNull())|
               (F.col('inte_sport_golf_v1_sport_golf_v1_cat').isNotNull())|
               (F.col('inte_sport_mixsport_v1_sport_mixsport_v1_cat').isNotNull())|
               (F.col('inte_healthcare_health_conditions_and_concerns_v1_healthcare_health_conditions_and_concerns_v1_cat').isNotNull())|
               (F.col('inte_healthcare_content_healthcare_content_cat').isNotNull())|
               (F.col('inte_healthcare_hospital_v1_healthcare_hospital_v1_cat').isNotNull())|
               (F.col('inte_healthcare_nutrition_and_fittness_v1_healthcare_nutrition_and_fittness_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[4]: Parents
        F.when((F.col('inte_lifestage_parents_v2_lifestage_parents_v2_cat').isNotNull())|
               (F.col('inte_lifestage_pregnancy_v1_lifestage_pregnancy_v1_cat').isNotNull())|
               (F.col('inte_lifestage_kids_5_8yo_v1_lifestage_kids_5_8yo_v1_cat').isNotNull())|
               (F.col('inte_lifestage_baby_v1_lifestage_baby_v1_cat').isNotNull())|
               (F.col('inte_lifestage_toddler_v1_lifestage_toddler_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[5]: University and International students
        F.when((F.col('inte_education_university_student_v1_education_university_student_v1_cat').isNotNull())|
               (F.col('inte_education_international_school_v1_education_international_school_v1_cat').isNotNull())|
               (F.col('inte_education_university_website_v1_education_university_website_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[6]: Ecommerce + Financial and Investment
        F.when((F.col('inte_ecommerce_consumer_electronics_brand_v1_ecommerce_consumer_electronics_brand_v1_cat').isNotNull())|
               (F.col('inte_ecommerce_consumer_electronics_retail_v1_ecommerce_consumer_electronics_retail_v1_cat').isNotNull())|
               (F.col('beha_credit_card_app_usage_v1_credit_card_app_usage_v1_cat').isNotNull())|
               (F.col('inte_finance_credit_v1_finance_credit_v1_cat').isNotNull())|
               (F.col('inte_insurance_insurance_nonlife_v2_insurance_insurance_nonlife_v2_cat').isNotNull())|
               (F.col('inte_securities_broker_site_v1_securities_broker_site_v1_cat').isNotNull())|
               (F.col('inte_securities_broker_site_v2_securities_broker_site_v2_cat').isNotNull())|
               (F.col('inte_realestate_real_estate_developer_v3_realestate_real_estate_developer_v3_cat').isNotNull()), 1).otherwise(0),
        # flag[7]: Cinema + Live Concerts
        F.when((F.col('inte_movie_ticket_v1_movie_ticket_v1_cat').isNotNull())|
               (F.col('inte_entertainment_cinema_v1_entertainment_cinema_v1_cat').isNotNull())|
               (F.col('inte_concert_ticket_v1_concert_ticket_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[8]: Pet Lovers
        F.when((F.col('inte_pet_lover_v1_pet_lover_v1_cat').isNotNull()), 1).otherwise(0)
    )
)

In [0]:
df_intermediate_interest = df_intermediate_interest.withColumn("interest_all", 
    F.array(
        # flag[0]: Fashion
        F.when(F.col("inte_fashion_apparel_v1_fashion_apparel_v1_cat").isNotNull(), 1).otherwise(0),  
        # flag[1]: Cosmetic Surgery 
        F.when(F.col('inte_cosmetic_surgery_v1_cosmetic_surgery_v1_cat').isNotNull(), 1).otherwise(0),     
        # flag[2]: Cosmetic
        F.when((F.col('inte_cosmetic_surgery_v1_cosmetic_surgery_v1_cat').isNotNull())|
               (F.col('inte_fashion_cosmetic_store_v2_fashion_cosmetic_store_v2_cat').isNotNull())|
               (F.col('inte_cosmetic_online_store_v1_cosmetic_online_store_v1_cat').isNotNull())|
               (F.col('inte_cosmetic_store_v1_cosmetic_store_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[3]: Ecommerce
        # 'inte_ecommerce_consumer_electronics_brand_v1_ecommerce_consumer_electronics_brand_v1_cat', # Ecommerce
        # 'inte_ecommerce_consumer_electronics_retail_v1_ecommerce_consumer_electronics_retail_v1_cat', # Ecommerce
        F.when((F.col('inte_ecommerce_consumer_electronics_brand_v1_ecommerce_consumer_electronics_brand_v1_cat').isNotNull())|
               (F.col('inte_ecommerce_consumer_electronics_retail_v1_ecommerce_consumer_electronics_retail_v1_cat').isNotNull()), 1).otherwise(0), 
        # flag[4]: F&B
        F.when((F.col('inte_food_chain_restuarant_v1_food_chain_restuarant_v1_cat').isNotNull())|
               (F.col('inte_food_junkfood_lover_v3_food_junkfood_lover_v3_cat').isNotNull())|
               (F.col('inte_chain_restuarant_chain_restuarant_cat').isNotNull()), 1).otherwise(0), 
        # flag[5]: Highend F&B
        F.when((F.col('inte_highend_japanese_food_v1_highend_japanese_food_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[6]: Hotel
        F.when((F.col('inte_luxury_hotel_thailand_v1_luxury_hotel_thailand_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[7]: Parents
        F.when((F.col('inte_lifestage_parents_v2_lifestage_parents_v2_cat').isNotNull())|
               (F.col('inte_lifestage_pregnancy_v1_lifestage_pregnancy_v1_cat').isNotNull())|
               (F.col('inte_lifestage_kids_5_8yo_v1_lifestage_kids_5_8yo_v1_cat').isNotNull())|
               (F.col('inte_lifestage_baby_v1_lifestage_baby_v1_cat').isNotNull())|
               (F.col('inte_lifestage_toddler_v1_lifestage_toddler_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[8]: Gym users  
        F.when((F.col('inte_healthcare_gym_v1_healthcare_gym_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[9]: Sports
        F.when((F.col('inte_sport_bicycle_v1_sport_bicycle_v1_cat').isNotNull())|
               (F.col('inte_sport_running_v1_sport_running_v1_cat').isNotNull())|
               (F.col('inte_sport_badminton_v1_sport_badminton_v1_cat').isNotNull())|
               (F.col('inte_sport_basketball_v1_sport_basketball_v1_cat').isNotNull())|
               (F.col('inte_basketball_content_basketball_content_cat').isNotNull())|
               (F.col('inte_sport_boxing_v1_sport_boxing_v1_cat').isNotNull())|
               (F.col('inte_sport_shooting_v1_sport_shooting_v1_cat').isNotNull())|
               (F.col('inte_sport_tennis_v1_sport_tennis_v1_cat').isNotNull())|
               (F.col('inte_sport_volleyball_v1_sport_volleyball_v1_cat').isNotNull())|
               (F.col('inte_sport_fishing_v1_sport_fishing_v1_cat').isNotNull())|
               (F.col('inte_sport_football_lover_v2_sport_football_lover_v2_cat').isNotNull())|
               (F.col('inte_sport_golf_v1_sport_golf_v1_cat').isNotNull())|
               (F.col('inte_sport_mixsport_v1_sport_mixsport_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[10]: Cinema
        F.when((F.col('inte_movie_ticket_v1_movie_ticket_v1_cat').isNotNull())|
               (F.col('inte_entertainment_cinema_v1_entertainment_cinema_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[11]: Pet Lovers
        F.when((F.col('inte_pet_lover_v1_pet_lover_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[12]: Health and Wellness
        F.when((F.col('inte_healthcare_health_conditions_and_concerns_v1_healthcare_health_conditions_and_concerns_v1_cat').isNotNull())|
               (F.col('inte_healthcare_content_healthcare_content_cat').isNotNull())|
               (F.col('inte_healthcare_hospital_v1_healthcare_hospital_v1_cat').isNotNull())|
               (F.col('inte_healthcare_nutrition_and_fittness_v1_healthcare_nutrition_and_fittness_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[13]: Live Concerts
        # bug fix: inte_concert_ticket_concert_ticket_cat
        F.when((F.col('inte_concert_ticket_v1_concert_ticket_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[14]: University and International students
        F.when((F.col('inte_education_university_student_v1_education_university_student_v1_cat').isNotNull())|
               (F.col('inte_education_international_school_v1_education_international_school_v1_cat').isNotNull())|
               (F.col('inte_education_university_website_v1_education_university_website_v1_cat').isNotNull()), 1).otherwise(0),
        # flag[15]: Financial and Investment
        F.when((F.col('beha_credit_card_app_usage_v1_credit_card_app_usage_v1_cat').isNotNull())|
               (F.col('inte_finance_credit_v1_finance_credit_v1_cat').isNotNull())|
               (F.col('inte_insurance_insurance_nonlife_v2_insurance_insurance_nonlife_v2_cat').isNotNull())|
               (F.col('inte_securities_broker_site_v1_securities_broker_site_v1_cat').isNotNull())|
               (F.col('inte_securities_broker_site_v2_securities_broker_site_v2_cat').isNotNull())|
               (F.col('inte_realestate_real_estate_developer_v3_realestate_real_estate_developer_v3_cat').isNotNull()), 1).otherwise(0),
        # flag[16]: Chinese
        F.when((F.col('inte_chinese_web_v1_chinese_web_v1_cat').isNotNull()), 1).otherwise(0)   
    )
)

# debug
# display(df_intermediate_interest.select('interest').limit(10))

In [0]:
df_intermediate_pay = df_intermediate_interest.withColumnRenamed('demo_affluence_grid_v1_affluence_group_bin','monthly_pay')
df_intermediate_pay = df_intermediate_pay.withColumn('monthly_pay', F.when(F.col('monthly_pay').isNull(), F.lit('unidentified'))
                                                .otherwise(F.col('monthly_pay'))
)

# display(df_intermediate_pay.select('monthly_pay').distinct())

In [0]:
# nationality
df_intermediate_country = df_intermediate_pay.withColumn("nationality", 
    F.when(F.col("a_country_name").isNull(), F.lit('nationality_unidentified'))
    .otherwise(F.col("a_country_name"))
)


# foreigner_type
df_intermediate_country = df_intermediate_country\
    .withColumn('foreigner_type', 
                F.when((F.col('demo_tourist_sim_v1_tourist_bin') == 'y'), F.lit('sim_tourist'))
                .when(F.col('roaming_flag')==1, F.lit('roaming'))
    .otherwise(F.lit('native')))

# debug
# display(df_intermediate_country.select('demo_tourist_sim_v1_tourist_bin','nationality','foreigner_type').distinct())

df_nation = (spark.read
      .option("header", "true").option("inferSchema", "true")
      .csv(nantional_path))

df_intermediate_country = df_intermediate_country.join(F.broadcast(df_nation), df_intermediate_country.nationality == df_nation.country_name, 'left')
df_intermediate_country = df_intermediate_country.withColumnRenamed('name','mall')
df_intermediate_country = df_intermediate_country.withColumnRenamed('Group','nationality_group')

# impute null with 'U'
df_intermediate_country = df_intermediate_country.fillna('U', subset=['nationality_group'])

# debug
# display(df_intermediate_country.select('nationality','nationality_group').distinct())

In [0]:
df_all_columns = df_intermediate_country.withColumnRenamed('home_region','region')
df_all_columns = df_all_columns.withColumnRenamed('day_type_final','day_type')
df_all_columns = df_all_columns.withColumn('interest_IS', F.col("interest_IS").cast("string"))
df_all_columns = df_all_columns.withColumn('interest_IL', F.col("interest_IL").cast("string"))
df_all_columns = df_all_columns.withColumn('interest_IF', F.col("interest_IF").cast("string"))
df_all_columns = df_all_columns.withColumn('interest_I', F.col("interest_I").cast("string"))
df_all_columns = df_all_columns.withColumn('interest_all', F.col("interest_all").cast("string"))

In [0]:
df_final = df_all_columns.select(
    'msisdn'
    ,'home_province'
    ,'a_country_name'
    ,'home_subdistrict'
    ,'home_district'
    ,'work_province'
    ,'work_subdistrict'
    ,'work_district'
    ,'demo_tourist_sim_v1_tourist_bin'
    ,'roaming_flag'
    ,'gender'
    ,'monthly_pay'
    ,'age_range'
    ,'region'
    ,'interest_IS'
    ,'interest_IL'
    ,'interest_IF'
    ,'interest_I'
    ,'interest_all'
    ,'nationality'
    ,'foreigner_type'
    ,'nationality_group'
    ,'is_bmr'
    ,'is_non_bmr'
    ,'is_foriegner'
).distinct()

In [0]:
save_to_parquet(df_final, save_path)